# What's in a Word: Breaking Texts and Making Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-Madison-Digital-Scholarship-Hub/text-and-data/blob/main/03-Tokenization/Whats-in-a-Word.ipynb)

Before text can be analyzed computationally, it has to be taken apart—split into pieces, normalized, stripped of noise. 

In this notebook, we'll introduce the preprocessing behind nearly all text analysis: tokenization, lemmatization, stemming, and stop word removal. We'll also distinguish this sense of "token" from how the word is used in large language models, a difference that reveals a lot about both. Our goal is to understand not just how to prepare text, but what each choice quietly decides.

> **Important Step**

Run the cells in order from top to bottom. Each cell depends on the ones above it. You don't need a GPU for this notebook; Colab's default CPU runtime is fine.

In [ ]:
""" If you're working in Colab, run this cell to bring everything, including the
saved copies of today's books, over from the GitHub repository. It's safe to re-run."""
import os # lets us check which folder we're in
if not os.path.exists('Whats-in-a-Word.ipynb'): # already in the right folder? then there's nothing to do
    if not os.path.exists('text-and-data'): # only download the repository once
        !git clone https://github.com/UW-Madison-Digital-Scholarship-Hub/text-and-data.git
    %cd text-and-data/03-Tokenization

## Packages
We're going to work with a set of Python packages for helping us access and manipulate textual data, including: 
- [**nltk**](https://www.nltk.org/): The Natural Language Toolkit, one of the most common packages for learning natural language processing
- [**spaCy**](https://spacy.io/): A faster, production oriented package for NLP tasks
- [**pandas**](https://pandas.pydata.org/): An open source data manipulation and analysis package
- [**gutenbergpy**](https://pypi.org/project/gutenbergpy/): A library for filtering, downloading, and querying Project Gutenberg data

In [ ]:
# Packages to install if necessary
# nltk, spaCy, pandas, and matplotlib come preinstalled in Colab, but gutenbergpy and tiktoken don't
%pip install -q nltk spacy pandas matplotlib gutenbergpy tiktoken

# spaCy also needs a trained English model, so we download it if it isn't already installed
import spacy
if not spacy.util.is_package("en_core_web_sm"):
    spacy.cli.download("en_core_web_sm")
import nltk

## What's a Word?

Most computational text analysis methods begin at the level of the word or, in the parlance of Natural Language Processing (NLP), the **token**. To a computer, a word is just a sequence of characters, stored as bits (or, rather, bytes) of code with no inherent meaning. But by manipulating, counting, and analyzing those sequences, we can find patterns that reveal everything from style and sentiment to the portrayal of gender across a collection of texts.

### To a Computer, What's a Word?
Take the word "count." To us, it's a word with several possible meanings. To a computer, it's five numbers, one for each character, stored as bytes. We can write those numbers in binary (the 1s and 0s the computer actually stores) or in hexadecimal (a shorter way of writing the same values):

| | c | o | u | n | t |
|---|---|---|---|---|---|
| **Binary** | 01100011 | 01101111 | 01110101 | 01101110 | 01110100 |
| **Hexadecimal** | 63 | 6F | 75 | 6E | 74 |

✏️ **Try it:** Run the cell below, then change the word. Try "Count" with a capital C, a word with an accent like "café," or your own name.

In [ ]:
word = "count" # ✏️ try "Count", "café", or your own name

for character in word:
    code = character.encode("utf-8") # the bytes the computer actually stores for this character
    binary = " ".join(f"{byte:08b}" for byte in code) # each byte written as 1s and 0s
    print(f"{character} -> hexadecimal: {code.hex(' ').upper():8} binary: {binary}")

A few things to notice:
- **The computer has no idea these numbers spell a word**, let alone which meaning of "count" we intend (to count apples, Count Dracula, or the count in a baseball game). All three are the same five bytes. Working out which is which takes context, which is what we'll use part-of-speech tagging for later on.
- **"Count" and "count" are different.** A capital C is `43`, not `63`, so to a computer these are two different strings. That's why we'll often lowercase text before counting words, which is one of many small choices we'll make along the way.
- **One character isn't always one byte.** The "é" in "café" takes two bytes (`C3 A9`), and characters in many other writing systems take three or four. Keep this in mind for the end of the notebook, where we'll see how large language models break text into pieces.

## Tokenization

The first step is breaking a string of text into its component pieces, or **tokens**. Depending on our approach, tokens might be whole words only, or they might include punctuation or pieces of words (like splitting "don't" into "do" and "n't").

In Python, our text starts out as a `string`: a sequence of characters, which could be `'It was the best of times'` or the entire text of *A Tale of Two Cities*. Tokenizing turns that one string into a `list` of smaller strings, one per token, which are much easier to count and analyze.

We'll start with the *Natural Language Toolkit*, or `nltk` to turn a string into a set of tokens. 

In [ ]:
# Import the packages and modules needed for tokenization
import nltk # import the Natural Language Toolkit
nltk.download('punkt_tab') # import the tokenization dataset from NLTK
from nltk.tokenize import word_tokenize # import the word tokenization module

In [ ]:
open0 = 'It was the best of times, it was the worst of times.'
tokens0 = word_tokenize(open0)
print(tokens0)

In [ ]:
open1 = 'Call me Ishmael.'
tokens1 = word_tokenize(open1)
print(tokens1)

✏️ **Try it:** Change the sentence below and see how NLTK splits it. Try contractions ("can't"), hyphenated words, abbreviations ("Dr."), numbers, or the first line of a favorite book.

In [ ]:
my_open = "It was a bright cold day in April, and the clocks were striking thirteen." # ✏️ change this sentence
print(word_tokenize(my_open))

## Stemming and Lemmatization

Stemming and Lemmatization are Natural Language Processing techniques for reducing derived or inflected words into a common base form for tasks such as counting. 

**Stemming** uses some basic heuristics to reduce common word forms to their root form or "stem." Note that not all stems are valid word forms (as we'll see). 
**Lemmatization** uses more sophisticated rules to return a valid dictionary form of a root word or "lemma."

For instance, imagine a sentence like "He thought the Lakeshore Path was his favorite run of all the runs he had done this year." 

We have the words "run" and "runs" as the singular and plural forms of the same noun. If we were counting words, we'd likely want to count them as versions of the same word. This is where stemming and lemmatization come in as processes to simplify word forms.

Let's look at an example.

In [ ]:
sent0 = 'The running man thought the Lakeshore Path was his favorite run of all the runs he had done this year'
sent_tokens0 = word_tokenize(sent0) #tokenize the sentence into a list of tokens

from nltk.stem import PorterStemmer
porter = PorterStemmer()

for token in sent_tokens0:
    print(porter.stem(token))

We can see that this correctly simplified the nouns "run" and "runs" into the common stem of "run" but it has also reduced "running" (which, in "the running man," describes the man) to "run," which may or may not be what we want. It also chops other words in ways that are usable but crude ("wa," "hi," "thi").

Stemming is a quick and dirty shortening technique: fast and cheap, but less nuanced than lemmatization. Let's take a look at how lemmatization handles this sentence:

In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

print(lemmatizer.lemmatize('running'))

In [ ]:
for token in sent_tokens0:
    print(lemmatizer.lemmatize(token))

The lemmatizer leaves more words intact than the stemmer did ("his" stays "his," "this" stays "this," "Lakeshore" stays "Lakeshore"), but look closely at "was": it became "wa," the same mistake the stemmer made. Why? Without any other information, the WordNet lemmatizer assumes every word is a noun, so it reads "was" as the plural of a (nonexistent) noun "wa." For the same reason, it leaves "running," "thought," "had," and "done" untouched, since none of them look like plural nouns.

The lemmatizer needs to know what *kind* of word it's looking at, which is exactly what part-of-speech tagging gives us. There are reasons to use a stemmer over a lemmatizer in specific contexts such as search/information retrieval but, in general, for most computational analysis in English and other languages for which a lemmatizing package exists, the lemmatizer (with part-of-speech information) tends to be the better choice. 

✏️ **Try it:** Can you find words the stemmer ruins? Stemmers sometimes merge words that mean very different things, and sometimes fail to merge words that belong together. Add your own words to the list below.

In [ ]:
words_to_stem = ["university", "universe", "organization", "organ", "happiness", "happily", "running", "ran"] # ✏️ add your own words
for word in words_to_stem:
    print(word, "->", porter.stem(word))

## Part-of-Speech (POS) Tagging

When analyzing texts it can be useful to know what part of speech (e.g., noun, adjective, verb, etc.) a given token belongs to. This can help with disambiguating between different words that have the same spelling. For instance, we can have the word "count" as a: 
- Verb, as in "I'm going to count the number of apples." 
- Noun as a title, as in "Count Dracula" or the "Count" on Sesame Street
- Noun as a record of the process of counting, as in "the count was 0 and 1" in baseball

Understanding what part of speech a word takes is highly contextual and relies on its position in a sentence and relative to other words. We can use a part-of-speech tagging algorithm to categorize words based on their part of speech. 

NLTK's tagger uses the tags from the [Penn Treebank](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html). Here are some of the most common ones:

| Tag | Part of Speech | Examples |
|-----|----------------|----------|
| NN | Noun, singular or mass | fox, dog, table |
| NNP | Proper noun, singular | John, London, Apple |
| VBZ | Verb, 3rd person singular present | jumps, is |
| VBD | Verb, past tense | had, went, walked |
| JJ | Adjective | quick, brown, lazy |
| RB | Adverb | completely, quickly, not |
| DT | Determiner | the, a, that |
| IN | Preposition or subordinating conjunction | over, for, in |

In [ ]:
nltk.download('averaged_perceptron_tagger_eng') # download the tagging algorithm dataset

In [ ]:
tagged_sent0 = nltk.pos_tag(sent_tokens0) # assign tags to each of the tokens in the sentence
print(tagged_sent0)

### Lemmatizing with POS Tags
The NLTK lemmatizer works without part-of-speech tags, but it's more accurate with them, since a word's part of speech helps determine its lemma. There's one wrinkle: the WordNet lemmatizer doesn't understand Penn Treebank tags like `VBD`. It only knows four parts of speech (noun, verb, adjective, and adverb), so we first need to translate each tag into one of those.

In [ ]:
# WordNet only knows four parts of speech, so we map the Penn Treebank tags onto them
def penn_to_wordnet(tag):
    """Convert a Penn Treebank tag (e.g., 'VBD') to the WordNet code the lemmatizer expects."""
    if tag.startswith('J'):
        return 'a' # adjective
    elif tag.startswith('V'):
        return 'v' # verb
    elif tag.startswith('R'):
        return 'r' # adverb
    else:
        return 'n' # noun (WordNet's default)

for token, tag in tagged_sent0:
    print(token, '->', lemmatizer.lemmatize(token, pos=penn_to_wordnet(tag)))


Now that the lemmatizer knows which words are verbs, "was," "thought," "had," and "done" are converted to their base (dictionary) forms: "be," "think," "have," and "do."

"Running" also becomes "run," just like it did with the stemmer. The tagger labeled it `VBG` (a verb form ending in "-ing"), so the lemmatizer treated it as a verb, even though in "the running man" it describes the man the way an adjective would. Whether "run" is the right result depends on what you're trying to count.

One key takeaway: the order of processing steps matters. The lemmatizer works better with part-of-speech tags, and the tagger works best on the full, original sentence, so we tag *before* we lemmatize, and we do both *before* removing stop words or anything else that changes the sentence.

✏️ **Try it:** Remember the different meanings of "count"? Let's see whether the tagger can tell them apart. Change the sentences, or add your own.

In [ ]:
count_sentences = [
    "I'm going to count the apples.",
    "Count Dracula counted the votes.",
    "The count was 0 and 1.",
] # ✏️ change these sentences or add your own

for sentence in count_sentences:
    print(nltk.pos_tag(word_tokenize(sentence)))

## Stop Words

Not all tokens are equally valuable for every kind of analysis. Nouns tend to tell us what a text is about and verbs tell us what's happening, but some extremely common words carry relatively little meaning on their own. We call these **stop words**, and removing them is a common step in cleaning up text for analysis. There's no single, definitive list of stop words for any language, but they commonly include articles ("a," "the"), prepositions ("in," "on"), and pronouns ("she," "he," "they").

Removing stop words can reduce the "noise" in a text, but what counts as noise depends on the question. If we were studying gendered language, for instance, the pronouns might be exactly what we need to keep.

Let's remove the stop words from the opening paragraph of *A Tale of Two Cities*:

In [ ]:
open_para = ' It was the best of times, it was the worst of times, it was the age of wisdom, it was the age of foolishness, it was the epoch of belief, it was the epoch of incredulity, it was the season of Light, it was the season of Darkness, it was the spring of hope, it was the winter of despair, we had everything before us, we had nothing before us, we were all going direct to Heaven, we were all going direct the other way—in short, the period was so far like the present period, that some of its noisiest authorities insisted on its being received, for good or for evil, in the superlative degree of comparison only.'

In [ ]:
from nltk.corpus import stopwords
nltk.download('stopwords')

In [ ]:
stop_words = set(stopwords.words('english')) # Select the English list of stopwords
tokens = word_tokenize(open_para)
filtered_tokens = [token for token in tokens if token.lower() not in stop_words]

print(filtered_tokens)

This isn't a very readable list, but it does foreground the more meaning-laden words in the paragraph: *wisdom*, *foolishness*, *belief*, *incredulity*, *hope*, *despair*. Notice what was lost, though: every "it was" and "we had" that gives the passage its rhythm and its voice. Whether that's noise or signal depends on the question we're asking, and the trade-off becomes clearer when we look at a whole book.

✏️ **Try it:** Which words are on NLTK's stop word list? Some of the answers may surprise you. Add your own words to check.

In [ ]:
words_to_check = ["not", "never", "no", "nobody", "she", "us", "against", "very"] # ✏️ add your own words
for word in words_to_check:
    print(word, "->", "stop word (removed)" if word in stop_words else "kept")

## Book-Length Example

Looking at the level of a sentence, it's clear that stop words carry relatively little meaning (in most contexts, at least). But what about at the level of a book? Let's take a look at a few examples. 

We'll use the `gutenbergpy` library, a library for querying and downloading books directly from Project Gutenberg. It allows us to pull the text of books from Project Gutenberg using the book's ID number, which can be found at the end of the URL for a specific book (e.g., the URL for one version of *Moby-Dick* is `https://www.gutenberg.org/ebooks/2701` and its ID number is `2701`).

One of the nice things about this particular package is that it allows you to download the text of the book without the Project Gutenberg headings and additional information, which saves a cleaning step. 

In [ ]:
# Let's start by getting a book from Project Gutenberg
import gutenbergpy.textget # Import the module that allows us to download the texts

dickens_id = 98 # ID for the book that we want to select.
dickens_raw = gutenbergpy.textget.get_text_by_id(dickens_id) # book, with header and license
dickens_clean = gutenbergpy.textget.strip_headers(dickens_raw) # removes headers and license

dickens = dickens_clean.decode('utf-8') # matches encoding to ensure clean text
print(dickens[:200]) # prints first 200 characters of the text

We can start with counting up the total number of words in the text and looking at the number of words that appear most frequently. We'll use the `pandas` library and simply tally the number of times that individual words appear. 

In [ ]:
# We'll create a function that will count the words in a given text
import re #regular expression library that enables manipulating strings of text
from collections import Counter
import pandas as pd

def word_counts(text):
    """Return a DataFrame of unique words and their counts, most frequent first."""
    # gutenbergpy returns bytes, so decode if needed
    if isinstance(text, bytes):
        text = text.decode("utf-8")

    # swap curly apostrophes for straight ones so "don’t" and "don't" count as the same word
    text = text.lower().replace("’", "'")

    # pull out words (keeping contractions like "don't")
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text)

    counts = Counter(words)
    df = pd.DataFrame(counts.most_common(), columns=["word", "count"])
    return df

In [ ]:
# Let's take a look at the top 20 most common words in A Tale of Two Cities
word_counts(dickens).head(20) # show the top 20 words in the list of words

In [ ]:
# We'll create a function that removes stop words from a given text
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True) # downloads NLTK's stop word lists if needed

def remove_stop_words(text, stop_words=None):
    """Return the text as a lowercase string of words with stop words removed."""
    # gutenbergpy returns bytes, so decode if needed
    if isinstance(text, bytes):
        text = text.decode("utf-8")

    # use NLTK's English stop word list unless we pass in our own
    if stop_words is None:
        stop_words = set(stopwords.words('english'))

    # swap curly apostrophes for straight ones so contractions match the stop word list
    text = text.lower().replace("\u2019", "'")

    # split into words the same way word_counts does, then drop the stop words
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text)
    kept = [word for word in words if word not in stop_words]

    return " ".join(kept) # join back into a string so it can go into word_counts

In [ ]:
# Now let's look at the top 20 words in A Tale of Two Cities without the stop words
dickens_no_stops = remove_stop_words(dickens)
word_counts(dickens_no_stops).head(20)

### Tagging the Book with spaCy
Counting words tells us *what* words appear, but not *how* they're being used. Part-of-speech (POS) tagging labels each word with its grammatical role (noun, verb, pronoun, etc.), and named entity recognition (NER) picks out names of people, places, organizations, and so on.

We'll use `spaCy` for this. Note that we run spaCy on the **original** text, not the stop-word-free version: the tagger relies on the surrounding words (including stop words) and capitalization to decide what each word is doing.

In [ ]:
# We'll create a function that runs spaCy over a whole book
import spacy

# load spaCy's small English model (we don't need the dependency parser, so we turn it off to save time)
nlp = spacy.load("en_core_web_sm", disable=["parser"])

def tag_text(text):
    """Run spaCy over the text one paragraph at a time and return a list of processed Docs."""
    # gutenbergpy returns bytes, so decode if needed
    if isinstance(text, bytes):
        text = text.decode("utf-8")

    # split on blank lines into paragraphs, and join the line breaks inside each paragraph
    # (spaCy can't take a whole book at once, and paragraphs are a natural unit anyway)
    paragraphs = [" ".join(p.split()) for p in re.split(r"\n\s*\n", text) if p.strip()]

    return list(nlp.pipe(paragraphs, batch_size=50))

In [ ]:
# Tag A Tale of Two Cities (this takes a minute or so, since spaCy is reading the whole book)
dickens_docs = tag_text(dickens)

In [ ]:
# We'll create a function that puts every word and its tags into a DataFrame
def pos_table(docs):
    """Return a DataFrame with one row per word: the word, its part of speech, its detailed tag, and its lemma."""
    rows = []
    for doc in docs:
        for token in doc:
            if token.is_alpha: # skip punctuation, numbers, and whitespace
                rows.append({
                    "word": token.text,
                    "pos": token.pos_, # broad part of speech (NOUN, VERB, PRON, ...)
                    "tag": token.tag_, # more detailed tag (e.g., VBD = past tense verb)
                    "lemma": token.lemma_,
                })
    return pd.DataFrame(rows)

dickens_pos = pos_table(dickens_docs)
dickens_pos.head(20)

### Counting Parts of Speech and Places
Now that every word is tagged, we can count just the words in a particular category. spaCy uses the [Universal POS tags](https://universaldependencies.org/u/pos/): `PRON` for pronouns, `VERB` for verbs, `NOUN` for common nouns, `PROPN` for proper nouns, and so on. Note that spaCy tags helping verbs like *was* and *had* (as in "had gone") as `AUX` rather than `VERB`.

For places, we'll use the entity labels `GPE` (countries, cities, states), `LOC` (other locations, like rivers or mountains), and `FAC` (buildings, bridges, streets).

In [ ]:
# We'll create a function that counts the words with a given part of speech
def pos_counts(docs, pos):
    """Return a DataFrame of words with the given part(s) of speech and their counts, most frequent first."""
    # accept either a single tag ("NOUN") or a list of tags (["NOUN", "PROPN"])
    if isinstance(pos, str):
        pos = [pos]

    words = [token.text.lower() for doc in docs for token in doc
             if token.pos_ in pos and token.is_alpha]

    counts = Counter(words)
    return pd.DataFrame(counts.most_common(), columns=["word", "count"])

In [ ]:
# Pronouns in A Tale of Two Cities
pos_counts(dickens_docs, "PRON").head(20)

In [ ]:
# Verbs in A Tale of Two Cities
pos_counts(dickens_docs, "VERB").head(20)

In [ ]:
# Nouns in A Tale of Two Cities
pos_counts(dickens_docs, "NOUN").head(20)

In [ ]:
# We'll create a function that counts the named entities with given labels (by default, places)
def entity_counts(docs, labels=("GPE", "LOC", "FAC")):
    """Return a DataFrame of named entities with the given labels and their counts, most frequent first."""
    entities = [ent.text for doc in docs for ent in doc.ents if ent.label_ in labels]

    counts = Counter(entities)
    return pd.DataFrame(counts.most_common(), columns=["entity", "count"])

# Places in A Tale of Two Cities
entity_counts(dickens_docs).head(20)

Take a look through these tables: does the tagger get everything right? Models like spaCy's are trained mostly on modern text (news, web pages), so 19th-century prose can trip them up. Watch for words tagged with the wrong part of speech, and for "places" that are actually people (or vice versa).

### Lemmatization
This is the same idea as the NLTK lemmatizer we used earlier, with part-of-speech tagging built in. spaCy already worked out each word's lemma (using its tag) when it tagged the book, so *was* and *were* become *be*, *hands* becomes *hand*, and *said* becomes *say*. We just need to collect the lemmas and join them back into a single string so that they can be fed into `remove_stop_words` and `word_counts`, just like the original text.

In [ ]:
# We'll create a function that turns tagged text into a string of lemmas
def lemmatize(docs):
    """Return the text as a lowercase string of lemmas, ready for remove_stop_words or word_counts."""
    lemmas = [token.lemma_.lower() for doc in docs for token in doc if token.is_alpha]
    return " ".join(lemmas)

dickens_lemmas = lemmatize(dickens_docs)
print(dickens_lemmas[:200])

In [ ]:
# Compare these to the top 20 words (without stop words) from before
word_counts(remove_stop_words(dickens_lemmas)).head(20)

### Putting It All Together
Now we can combine everything into a single workflow: download a book, tag it once with spaCy, and build all of our tables from that one pass. The function returns a dictionary of DataFrames, so we can look at whichever table we want by name.

In [ ]:
# We'll create a function that runs the whole workflow on a Project Gutenberg book
def analyze_book(book_id):
    """Download a Project Gutenberg book and return a dictionary of word frequency tables."""
    # 1. download the book and strip the Project Gutenberg header and license
    raw = gutenbergpy.textget.get_text_by_id(book_id)
    text = gutenbergpy.textget.strip_headers(raw).decode("utf-8")

    # 2. tag the book with spaCy (the slow step, so we only do it once)
    docs = tag_text(text)

    # 3. build each table from the original text or the tagged docs
    return {
        "words": word_counts(text),
        "words_no_stops": word_counts(remove_stop_words(text)),
        "lemmas_no_stops": word_counts(remove_stop_words(lemmatize(docs))),
        "pronouns": pos_counts(docs, "PRON"),
        "verbs": pos_counts(docs, "VERB"),
        "nouns": pos_counts(docs, "NOUN"),
        "places": entity_counts(docs),
    }

In [ ]:
# Let's try it out on Moby-Dick
moby = analyze_book(2701)

# show the top 10 of each table side by side
pd.concat({name: table.head(10) for name, table in moby.items()}, axis=1)

### Comparing Books: Pronouns
Pronouns are a good example of why we might *not* want to throw stop words away: they tell us who a book is about and who is speaking. Let's put the top 20 pronouns from *A Tale of Two Cities* and *Moby-Dick* side by side.

There's one catch: *Moby-Dick* is a much longer book (about 217,000 words versus about 137,000), so its raw counts will be bigger just because there are more words. To compare the two fairly, we'll also convert each count into a rate: how many times the pronoun appears per 1,000 words.

In [ ]:
# We'll create a function that adds a "per 1,000 words" rate to a table of counts
def per_1000_words(counts, words, n=20):
    """Return the top n rows of a counts table with each count converted to a rate per 1,000 words in the book."""
    total_words = words["count"].sum() # total number of words in the book
    table = counts.head(n).copy()
    table["per_1000_words"] = (table["count"] / total_words * 1000).round(1)
    return table

# Top 20 pronouns in A Tale of Two Cities and Moby-Dick, side by side
pd.concat({
    "A Tale of Two Cities": per_1000_words(pos_counts(dickens_docs, "PRON"), word_counts(dickens)),
    "Moby-Dick": per_1000_words(moby["pronouns"], moby["words"]),
}, axis=1)

Tables are precise, but a chart makes the differences easier to see at a glance. Let's plot a handful of pronouns for both books:

In [ ]:
# We'll create a function that draws a bar chart of pronoun rates for up to three books
import matplotlib.pyplot as plt

BOOK_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"] # blue, orange, aqua: easy to tell apart, including for colorblind readers

def plot_pronoun_rates(books, pronouns=("i", "we", "you", "he", "she", "they", "his", "her")):
    """Draw a bar chart comparing how often up to three books use the given pronouns (per 1,000 words)."""
    # books is a dictionary like {"Moby-Dick": (pronoun_counts, word_counts)}
    rates = {}
    for name, (pronoun_counts, words) in books.items():
        table = per_1000_words(pronoun_counts, words, n=len(pronoun_counts)) # rates for every pronoun, not just the top 20
        rates[name] = table.set_index("word")["per_1000_words"].reindex(list(pronouns)).fillna(0)
    rates = pd.DataFrame(rates)

    ax = rates.plot.barh(figsize=(8, 6), width=0.8, color=BOOK_COLORS[:len(books)], edgecolor="white", linewidth=1)
    ax.invert_yaxis() # list the pronouns from top to bottom in the order we gave them
    ax.set_title("How often does each book use these pronouns?", loc="left")
    ax.set_xlabel("Times per 1,000 words")
    ax.set_ylabel("")
    ax.spines[["top", "right"]].set_visible(False) # remove the box around the chart
    ax.grid(axis="x", color="#e5e5e5") # light gridlines to help read the values
    ax.set_axisbelow(True)
    ax.legend(frameon=False)
    plt.show()

plot_pronoun_rates({
    "A Tale of Two Cities": (pos_counts(dickens_docs, "PRON"), word_counts(dickens)),
    "Moby-Dick": (moby["pronouns"], moby["words"]),
})

What do these lists tell us about the two books? A few things to look for:
- **Who's in the book?** "She" doesn't make the top 20 for *Moby-Dick* at all, and "her" appears about five times less often than in *A Tale of Two Cities* (1.5 versus 7.6 times per 1,000 words). And many of the "her"s in *Moby-Dick* aren't women at all, but ships and boats.
- **Who's speaking?** "We" and "us" show up in *Moby-Dick*'s list but not in Dickens', which hints at a narrator who is one member of a crew.
- **How much is story, and how much is something else?** *Moby-Dick* uses fewer pronouns overall. Long stretches of the book are essays about whales and whaling rather than scenes between characters.
- **What did the tagger miss?** *Moby-Dick* is full of "thou," "thee," and "ye," but you won't find "thou" anywhere in its pronoun list (spaCy tagged it as something else, and it even showed up in our list of places). A model trained on modern English doesn't always recognize older forms, so its mistakes shape our counts.

## Your Turn
Now it's your turn to make the choices. The activities below walk through each step from this notebook (tokenizing, stemming and lemmatizing, removing stop words, and tagging) using a book that you pick. Work through them in order, since each one builds on the one before it.

Look for the ✏️ in the code cells: that marks a line you can change. Then use the questions under each activity to think about what your choices decided.

**Before you start:** run the catch-up cell below. If you've run every cell above, it won't do anything. But if your session restarted (Colab disconnects after a while if it's left idle) or you skipped ahead, it reloads all of the functions and books that the activities need from `helpers.py`, a file in this folder with copies of the functions we built above.

*If Colab disconnected, first re-run the two setup cells at the very top of the notebook (the one that copies the GitHub repository and the one that installs the packages), then run the catch-up cell.*

In [ ]:
# Catch-up cell: restores everything the activities need if your session restarted or you skipped ahead
import os # lets us check that helpers.py is in this folder
needed = ["word_tokenize", "porter", "lemmatizer", "penn_to_wordnet", "stopwords", "word_counts",
          "remove_stop_words", "tag_text", "pos_table", "pos_counts", "entity_counts", "analyze_book",
          "per_1000_words", "plot_pronoun_rates", "dickens", "dickens_docs", "moby"]
missing = [name for name in needed if name not in globals()]

if missing:
    print("Catching up (this takes a minute or so, since we need to re-tag A Tale of Two Cities and Moby-Dick)...")
    if not os.path.exists("helpers.py"):
        raise FileNotFoundError("Can't find helpers.py. Re-run the two setup cells at the top of the notebook, then try this cell again.")
    from helpers import * # load all of our functions from helpers.py
    dickens = load_book(98)
    dickens_docs = tag_text(dickens)
    moby = analyze_book(2701)
    print("All caught up!")
else:
    print("You're all caught up, so there's nothing to do.")

### Activity 1: Pick a Book
Choose a book from the list below, or find another one on [Project Gutenberg](https://www.gutenberg.org/) and use the ID number at the end of its URL. The books in this list are already saved in this folder, so they'll load instantly; any other book will be downloaded from Project Gutenberg.

| ID | Title | Author |
|----|-------|--------|
| 1342 | *Pride and Prejudice* | Jane Austen |
| 1260 | *Jane Eyre* | Charlotte Brontë |
| 84 | *Frankenstein* | Mary Shelley |
| 345 | *Dracula* | Bram Stoker |
| 11 | *Alice's Adventures in Wonderland* | Lewis Carroll |
| 1661 | *The Adventures of Sherlock Holmes* | Arthur Conan Doyle |

In [ ]:
# Now you can grab a book of your choice
import gutenbergpy.textget # Import the module that allows us to download the texts

book_id = 0 # ✏️ Replace 0 with the ID number of the book you want (e.g., 1342 for Pride and Prejudice)
if book_id == 0:
    raise ValueError("Pick a book first: replace the 0 in 'book_id = 0' with a book's ID number.")

book_raw = gutenbergpy.textget.get_text_by_id(book_id) # book, with header and license
book_clean = gutenbergpy.textget.strip_headers(book_raw) # removes headers and license

book = book_clean.decode('utf-8') # matches encoding to ensure clean text
print(f"{len(book.split()):,} words") # roughly how long is the book?
print(book[:200]) # prints first 200 characters of the text

### Activity 2: What's a Token?
Copy a sentence from your book (or write your own tricky sentence) into `my_sentence` below. **Before you run the cell**, guess how many tokens `word_tokenize` will split it into.

- Did the count match your guess? Which tokens surprised you?
- Where do `word_tokenize` and splitting on spaces (`.split()`) disagree? Which one is "right"?
- Try adding contractions ("didn't"), hyphens ("well-known"), abbreviations ("Mr."), numbers, or dashes. How does each one get split?

In [ ]:
my_sentence = "Mr. Lorry didn't pay £3.50 for the ice-cream—or did he?" # ✏️ replace with a sentence from your book, or write your own

my_tokens = word_tokenize(my_sentence) # split the sentence into tokens with NLTK
print(len(my_tokens), "tokens with word_tokenize:")
print(my_tokens)

print(len(my_sentence.split()), "tokens when we just split on spaces:")
print(my_sentence.split())

### Activity 3: Stems vs. Lemmas
Now let's stem and lemmatize your sentence, side by side. The table also shows the part-of-speech tag that the lemmatizer used (the Penn Treebank table in the Part-of-Speech Tagging section above lists the most common tags).

- Find a word where the stem and the lemma disagree. If you were counting words, which version would you want?
- Did the tagger get any of the tags wrong? If so, what happened to that word's lemma?

In [ ]:
my_tagged = nltk.pos_tag(my_tokens) # tag each token with its part of speech

pd.DataFrame({
    "token": my_tokens,
    "tag": [tag for token, tag in my_tagged],
    "stem": [porter.stem(token) for token in my_tokens],
    "lemma": [lemmatizer.lemmatize(token, pos=penn_to_wordnet(tag)) for token, tag in my_tagged],
})

### Activity 4: Choose Your Stop Words
First, take a look at NLTK's full list of English stop words. Is there anything on it that surprises you? Are there words you'd want to keep for your own research?

In [ ]:
default_stop_words = set(stopwords.words('english')) # NLTK's English list of stop words
print(len(default_stop_words), "stop words:")
print(sorted(default_stop_words))

Here are the top 20 words in your book with NLTK's stop words removed. Are there words in this list that crowd out more meaningful ones (like "said," "mr," and "upon" in *A Tale of Two Cities*)?

In [ ]:
# Top 20 words in your book without NLTK's stop words
word_counts(remove_stop_words(book)).head(20)

Now make your own stop word list: add the words you'd like to remove, and put back any words from NLTK's list that you think matter.

- How did your changes reshape the list?
- Who might make a different choice than you did, and why? (Think of someone studying gender, negation, or dialogue.)

In [ ]:
my_stop_words = set(stopwords.words('english')) # start from NLTK's list
my_stop_words = my_stop_words | {"said", "mr", "upon"} # ✏️ add words you want to remove
my_stop_words = my_stop_words - {"not", "no"} # ✏️ put back words you want to keep

word_counts(remove_stop_words(book, stop_words=my_stop_words)).head(20)

### Activity 5: Tag Your Book
Next, we'll run spaCy over your book. This is the slow step (it can take a minute or so, depending on the length of your book), so we only do it once.

In [ ]:
book_docs = tag_text(book) # tag every word in the book with spaCy

Now we can count the words in any part of speech. Try a few different ones:
- `"NOUN"` (nouns), `"VERB"` (verbs), `"ADJ"` (adjectives), `"ADV"` (adverbs), or `"PROPN"` (proper nouns)

What does each list tell you about your book? What do the adjectives say about the world it describes? Do the proper nouns give you a cast of characters?

In [ ]:
pos_counts(book_docs, "ADJ").head(20) # ✏️ try "NOUN", "VERB", "ADV", or "PROPN"

### Activity 6: Catch the Tagger
Taggers make mistakes, and those mistakes end up in our counts. Pick a word that can be more than one part of speech (like "count," "light," "close," "watch," or "run") and see how spaCy tagged it in your book. The cell below shows how many times the word got each tag, along with a few passages so you can check whether the tagger got it right.

*Hint: if you picked* Dracula*, try "count."*

In [ ]:
my_word = "light" # ✏️ pick a word that can be more than one part of speech

# How many times did the word get each tag?
book_pos = pos_table(book_docs)
print(book_pos[book_pos["word"].str.lower() == my_word]["pos"].value_counts())
print()

# Show a few passages with the word, so we can check the tags
shown = 0
for doc in book_docs:
    for token in doc:
        if token.lower_ == my_word and shown < 8:
            start, end = max(token.i - 8, 0), min(token.i + 8, len(doc))
            print(f"[{token.pos_}] ...{doc[start:end].text}...")
            shown += 1

We can do the same thing with places. Look through the list below: are all of these really places? What kinds of mistakes is spaCy making?

In [ ]:
entity_counts(book_docs).head(20) # places in your book

### Activity 7: Compare Your Book
Finally, let's add your book to our pronoun comparison with *A Tale of Two Cities* and *Moby-Dick*, first as a table and then as a chart.

- Who's in your book? How do "he" and "she" (and "his" and "her") compare?
- Who's speaking? Is "I" near the top of the list (which might suggest a first-person narrator)? What about "you" or "we"?
- What would you have lost if you had removed the stop words before counting?

In [ ]:
# Top 20 pronouns in all three books, side by side
pd.concat({
    "A Tale of Two Cities": per_1000_words(pos_counts(dickens_docs, "PRON"), word_counts(dickens)),
    "Moby-Dick": per_1000_words(moby["pronouns"], moby["words"]),
    "Your book": per_1000_words(pos_counts(book_docs, "PRON"), word_counts(book)),
}, axis=1)

In [ ]:
plot_pronoun_rates({
    "A Tale of Two Cities": (pos_counts(dickens_docs, "PRON"), word_counts(dickens)),
    "Moby-Dick": (moby["pronouns"], moby["words"]),
    "Your book": (pos_counts(book_docs, "PRON"), word_counts(book)),
}) # ✏️ try changing which pronouns to compare by adding pronouns=("i", "we", "my", "our") inside the parentheses

## Tokens in Large Language Models
Large language models (LLMs) like ChatGPT, Claude, and Gemini also start by breaking text into **tokens**, but their tokens are something quite different from the ones we've been working with. An LLM's tokenizer isn't built from rules about English (like NLTK's tokenizer) or from a dictionary (like the lemmatizer). Instead, its vocabulary is *learned* from enormous amounts of text: common words get a token of their own, and rarer words get built out of smaller, frequent pieces. Each token is then turned into a number (its ID), and those numbers are what the model actually reads.

We'll use `tiktoken`, the tokenizer that OpenAI's GPT-4o model uses, to see how this works. (Other models, including Claude and Gemini, use their own tokenizers, but they work in similar ways.)

In [ ]:
import tiktoken # OpenAI's tokenizer library

encoding = tiktoken.get_encoding("o200k_base") # the vocabulary GPT-4o uses (it downloads the first time you run this)
print(f"This tokenizer's vocabulary has {encoding.n_vocab:,} tokens")

def llm_tokens(text):
    """Return the LLM tokens (as text) and their ID numbers for a string."""
    ids = encoding.encode(text) # turn the text into a list of token IDs
    tokens = [encoding.decode([i]) for i in ids] # turn each ID back into its piece of text
    return tokens, ids

In [ ]:
# Let's compare the LLM's tokens to NLTK's for the opening of A Tale of Two Cities
opening = "It was the best of times, it was the worst of times."
tokens, ids = llm_tokens(opening)
print("NLTK:", word_tokenize(opening))
print("LLM: ", tokens)
print("IDs: ", ids)

At first glance these look almost the same, but look closely at the LLM's tokens: most of them start with a space (`' was'`, `' the'`). The space is part of the token. And the tokens don't stay as text: the model only ever sees the ID numbers.

Now let's try some words that are less common:

In [ ]:
for text in ["Call me Ishmael.", "Ishmeal", "Defarge", "incredulity", "unbelievableness", "1859", "the The THE"]:
    tokens, ids = llm_tokens(text)
    print(f"{text!r:20} -> {tokens}")

A few things to notice:
- **Rare words get split into pieces.** "Ishmael" becomes "Ish" + "mael," and "Defarge" becomes "Def" + "arge." This is a little like stemming, except that the pieces are chosen by how often they appear in the training data, not by rules about English. Sometimes the pieces look like meaningful parts of a word ("in" + "cred" + "ul" + "ity"); sometimes they don't ("un" + "bel" + "iev" + "ab" + "leness").
- **Misspellings and numbers get chopped up too.** "Ishmeal" becomes "I" + "sh" + "meal," and "1859" becomes "185" + "9."
- **Nothing gets normalized.** "the," " The," and " THE" are three different tokens with three different IDs. There's no lowercasing, no stemming or lemmatizing, and no stop word removal: the model keeps everything, and learns from context which words matter.

Which words get their own token depends on what was common in the text the tokenizer learned from. That has consequences for languages, too. Here's roughly the same sentence in a few different languages:

In [ ]:
sentences = {
    "English": "It was the best of times.",
    "Spanish": "Era el mejor de los tiempos.",
    "Russian": "Это было лучшее из времён.",
    "Swahili": "Ilikuwa nyakati bora zaidi.",
}

for language, sentence in sentences.items():
    tokens, ids = llm_tokens(sentence)
    print(f"{language:8} {len(sentence.split())} words -> {len(tokens)} tokens: {tokens}")

English and Spanish words mostly get a token of their own, while Russian and Swahili words get split into more pieces. LLMs have limits on how much text they can read at once, and companies charge by the token, so text in languages that were less common in the training data takes up more of that limit and costs more to process.

How many tokens is a whole book?

In [ ]:
dickens_words = word_counts(dickens)["count"].sum()
dickens_tokens = len(encoding.encode(dickens))
print(f"A Tale of Two Cities: {dickens_words:,} words -> {dickens_tokens:,} LLM tokens ({dickens_tokens / dickens_words:.2f} tokens per word)")

✏️ **Try it:** Tokenize some text of your own. Try a sentence from your book, a word from your field, your name, or a sentence in another language you know.

In [ ]:
my_text = "Supercalifragilisticexpialidocious" # ✏️ change this text
tokens, ids = llm_tokens(my_text)
print(len(tokens), "tokens:", tokens)
print("IDs:", ids)

### Two Kinds of Tokens

| | Tokens in this workshop | Tokens in an LLM |
|---|---|---|
| **What's a token?** | A word or punctuation mark | A word, a piece of a word, or even a piece of a character |
| **Where do the rules come from?** | Rules about language (spaces, punctuation, contractions) | Patterns learned from the training data |
| **Capitals and spaces** | Usually lowercased; spaces are dropped | Kept: "The" and " the" are different tokens |
| **Stop words** | Often removed | Always kept |
| **What happens next?** | We count and compare them | They become numbers that the model reads |
| **Who makes the choices?** | You, the researcher | The people who built the model, before you ever see it |

That last row is the biggest difference. Every choice we made in this notebook (how to split words, whether to stem or lemmatize, which stop words to remove) was ours to make and to question. In an LLM, those choices have already been made, quietly, before the text ever reaches the model.

## Wrapping Up: What Did Each Choice Decide?
Every step in this notebook made a decision about what counts as a word, and which words count. Before you go, think back over your choices:

- **Tokenizing:** What did your tokenizer decide was (or wasn't) a word?
- **Stemming and lemmatizing:** Which words got merged together? Should they have been?
- **Stop words:** What did you throw away, and what would someone else have kept?
- **Tagging:** Where did the tagger get it wrong, and how might that change what you conclude about your book?
- **LLM tokens:** Who decided what counts as a token for an LLM? What might that mean for the languages, names, and words it handles well (or poorly)?

**Bonus:** Run `analyze_book()` on a second book from the list and compare its tables with your first book. What changes, and what stays the same?

In [ ]:
second_book = analyze_book(84) # ✏️ replace 84 with the ID of another book

# show the top 10 of each table side by side
pd.concat({name: table.head(10) for name, table in second_book.items()}, axis=1)